# 00 · Setup and tests

**Settings:** Accelerator *GPU T4 x2* (or None), Internet **on** (installs Qiskit for the hardware tests).
**Inputs:** the `bbd-code` dataset.

This notebook checks the environment, runs the full test suite and benchmarks the simulator on the GPU,
then estimates how many Kaggle GPU-hours the grid will need. It writes nothing that later notebooks need.

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
import platform, subprocess, torch, psutil
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))
print("CPUs", os.cpu_count(), "| RAM %.1f GB" % (psutil.virtual_memory().total / 2**30),
      "| disk free %.1f GB" % (shutil.disk_usage("/kaggle/working" if os.path.isdir("/kaggle/working") else ".").free / 2**30))

In [ ]:
# Pinned: these are the versions the hardware code was tested with (SamplerV2 API).
%pip install -q "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0" "qiskit-aer==0.18.0" pylatexenc

## Test suite
Simulator against a dense reference and Qiskit, gradients, Lemma 1 and Lemma 2, metrics, the data pipeline on synthetic files with the real schemas, tuning, grid resume and carry-forward, statistics, every figure, and the hardware pipeline on a simulated `ibm_kingston` (FakeKingston). Takes about 3–6 minutes.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "-W", "ignore", f"{CODE}/tests"],
                   capture_output=True, text=True, cwd=CODE)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "tests failed - do not start the grid"

## GPU benchmark and grid estimate

In [ ]:
from bbd import bench
n_workers = max(1, torch.cuda.device_count())
ms, est = bench.report(n_workers=n_workers)
print({k: f"{v:.2f} ms/step" for k, v in ms.items()})
print(f"grid: {est['n_grid']} runs ({est['n_quantum']} quantum) -> about {est['grid_hours']:.1f} h wall-clock on {n_workers} GPU(s)")
print(f"tuning: about {est['tuning_hours']:.1f} h.  Kaggle counts GPU quota per session-hour, so quota used ~ wall-clock.")